# 01 · Ingestion — from 583 MB of CSV to a columnar Spark dataset

The ECG Heartbeat Categorization dataset ships as four head-less CSV files: a matrix
of 188 float columns per row, where the first 187 numbers are the amplitude samples
of one segmented heartbeat and the last one is the class label.

This notebook turns those four files into a single canonical Spark DataFrame — one
row per heartbeat, the waveform held as an `array<double>`, a typed label and
provenance columns — and persists it as partitioned Parquet. Every later notebook
reads that Parquet copy instead of re-parsing text.

**Source:** [ECG Heartbeat Categorization Dataset](https://www.kaggle.com/datasets/shayanfazeli/heartbeat) ·
Kachuee, Fazeli & Sarrafzadeh, *ECG Heartbeat Classification: A Deep Transferable Representation* ([arXiv:1805.00794](https://arxiv.org/abs/1805.00794))

In [1]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Works from a fresh clone (no `pip install -e .` needed) and on Databricks,
# where the repo folder is added to the workspace instead of installed.
try:
    import ecg
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent / "src"))
    import ecg

print("ecg", ecg.__version__)

ecg 0.1.0


In [2]:
import time

import pandas as pd
from pyspark.sql import functions as F

from ecg import get_spark, load_config
from ecg.session import session_summary
from ecg.schema import (
    N_SAMPLES,
    N_COLUMNS,
    SAMPLING_RATE_HZ,
    MITBIH_LABEL_DESCRIPTIONS,
    raw_schema,
    canonical_schema,
)
from ecg.ingest import read_raw_csv, to_canonical, load_all, ingest_all, RAW_FILE_SPECS
from ecg.config import RAW_FILES

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## Configuration

`load_config()` resolves paths from `conf/config.yaml`, environment variables and
its own defaults. The raw CSVs are expected in the `data/` folder next to the
repository — 583 MB has no business inside a git working tree — and
`ECG_DATA_DIR` overrides that if they live elsewhere.

In [3]:
cfg = load_config()
cfg.validate_raw()

pd.DataFrame(
    [
        {
            "file": name,
            "collection": RAW_FILE_SPECS[key]["source"],
            "split": RAW_FILE_SPECS[key]["split"],
            "size_MB": round(cfg.raw_path(key).stat().st_size / 1024**2, 1),
        }
        for key, name in RAW_FILES.items()
    ]
)

,file,collection,split,size_MB
0,mitbih_train.csv,mitbih,train,392.4
1,mitbih_test.csv,mitbih,test,98.1
2,ptbdb_normal.csv,ptbdb,full,18.1
3,ptbdb_abnormal.csv,ptbdb,full,47.1


## Spark session

`get_spark()` builds a local session with a modest driver and Arrow enabled. On a
Databricks cluster the same call attaches to the existing session instead, so the
notebook runs unchanged in both places.

In [4]:
spark = get_spark(cfg)
spark.sparkContext.setLogLevel("ERROR")
session_summary(spark)

{'spark_version': '4.2.0',
 'application_id': 'local-1787827487080',
 'master': 'local[*]',
 'default_parallelism': 2,
 'shuffle_partitions': '16',
 'arrow_enabled': 'true',
 'databricks': False}

## Reading a raw file

An **explicit schema** is used rather than `inferSchema`. Inference would cost a
full extra pass over 583 MB of text to rediscover a layout the dataset authors
already documented: 187 sample columns plus one label column, all doubles.

In [5]:
schema = raw_schema()
print(f"{len(schema.fields)} columns  ({N_SAMPLES} samples + 1 label)")
print("first:", [f.name for f in schema.fields[:3]])
print("last :", [f.name for f in schema.fields[-3:]])

188 columns  (187 samples + 1 label)
first: ['s000', 's001', 's002']
last : ['s185', 's186', 'label_raw']


In [6]:
raw = read_raw_csv(spark, cfg.raw_path("mitbih_train"))
print(f"rows: {raw.count():,}   columns: {len(raw.columns)}")
raw.select("s000", "s001", "s002", "s185", "s186", "label_raw").show(5)

rows: 87,554   columns: 188


+------------------+------------------+-------------------+----+----+---------+
|              s000|              s001|               s002|s185|s186|label_raw|
+------------------+------------------+-------------------+----+----+---------+
|0.9779411554336548|0.9264705777168274| 0.6813725233078003| 0.0| 0.0|      0.0|
|0.9601139426231384|0.8632478713989258| 0.4615384638309479| 0.0| 0.0|      0.0|
|               1.0|0.6594594717025757|0.18648648262023926| 0.0| 0.0|      0.0|
|0.9254143834114075|0.6657458543777466|  0.541436493396759| 0.0| 0.0|      0.0|
|0.9671361446380615|               1.0| 0.8309859037399292| 0.0| 0.0|      0.0|
+------------------+------------------+-------------------+----+----+---------+
only showing top 5 rows


The label really is the last column, stored in scientific notation like every
other value — which is why the schema reads it as a double and the canonical form
casts it to an integer.

In [7]:
raw.groupBy("label_raw").count().orderBy("label_raw").show()

+---------+-----+
|label_raw|count|
+---------+-----+
|      0.0|72471|
|      1.0| 2223|
|      2.0| 5788|
|      3.0|  641|
|      4.0| 6431|
+---------+-----+



## The canonical form

187 wide columns collapse into a single `array<double>`. Two reasons: the whole
downstream analysis can then be written with Spark's higher-order array functions
(`transform`, `aggregate`, `filter`, `zip_with`) instead of 187 column references,
and the Parquet footprint drops sharply.

In [8]:
canonical_schema()

StructType([StructField('beat_id', StringType(), False), StructField('source', StringType(), False), StructField('split', StringType(), False), StructField('label', IntegerType(), False), StructField('label_name', StringType(), False), StructField('signal', ArrayType(DoubleType(), True), False)])

In [9]:
beats_train = to_canonical(raw, source="mitbih", split="train")
beats_train.select("beat_id", "source", "split", "label", "label_name", F.size("signal").alias("n_samples")).show(5, truncate=False)

+-------------------------+------+-----+-----+----------+---------+
|beat_id                  |source|split|label|label_name|n_samples|
+-------------------------+------+-----+-----+----------+---------+
|mitbih_train_000000000000|mitbih|train|0    |N         |187      |
|mitbih_train_000000000001|mitbih|train|0    |N         |187      |
|mitbih_train_000000000002|mitbih|train|0    |N         |187      |
|mitbih_train_000000000003|mitbih|train|0    |N         |187      |
|mitbih_train_000000000004|mitbih|train|0    |N         |187      |
+-------------------------+------+-----+-----+----------+---------+
only showing top 5 rows


In [10]:
example = beats_train.select("label_name", "signal").first()
print("class:", example["label_name"], "—", MITBIH_LABEL_DESCRIPTIONS[example["label_name"]])
print("first 8 samples:", [round(v, 4) for v in example["signal"][:8]])
print("last 8 samples :", [round(v, 4) for v in example["signal"][-8:]], "  <- zero padding")
print(f"beat duration if unpadded: {N_SAMPLES / SAMPLING_RATE_HZ:.2f} s at {SAMPLING_RATE_HZ} Hz")

class: N — Normal beat (normal, left/right bundle branch block, atrial escape, nodal escape)
first 8 samples: [0.9779, 0.9265, 0.6814, 0.2451, 0.1544, 0.1912, 0.152, 0.0858]
last 8 samples : [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]   <- zero padding
beat duration if unpadded: 1.50 s at 125 Hz


## All four files at once

`load_all` unions both collections. MIT-BIH keeps its published `train`/`test`
split; the PTB files carry no split of their own, so both are tagged `full` and a
reproducible split is created later, in notebook 04.

In [11]:
beats = load_all(spark, cfg)
beats.groupBy("source", "split").agg(
    F.count(F.lit(1)).alias("n_beats"),
    F.countDistinct("label").alias("n_classes"),
).orderBy("source", "split").show()

+------+-----+-------+---------+
|source|split|n_beats|n_classes|
+------+-----+-------+---------+
|mitbih| test|  21892|        5|
|mitbih|train|  87554|        5|
| ptbdb| full|  14552|        2|
+------+-----+-------+---------+



## Persisting as Parquet

The write is partitioned by `source` and `split`, so a notebook that only needs
MIT-BIH never reads a PTB byte.

In [12]:
started = time.time()
stored = ingest_all(spark, cfg, write=True)
total = stored.count()
elapsed = time.time() - started

csv_mb = sum(cfg.raw_path(key).stat().st_size for key in RAW_FILES) / 1024**2
parquet_mb = sum(
    path.stat().st_size for path in cfg.parquet_path("beats").rglob("*") if path.is_file()
) / 1024**2

print(f"{total:,} beats written in {elapsed:.1f}s")
print(f"CSV      {csv_mb:8.1f} MB")
print(f"Parquet  {parquet_mb:8.1f} MB   ({csv_mb / parquet_mb:.1f}x smaller)")

123,998 beats written in 11.5s
CSV         555.8 MB
Parquet      48.2 MB   (11.5x smaller)


In [13]:
# Round-trip check: the stored copy must match the source row for row.
reloaded = stored.groupBy("source", "split", "label_name").count().orderBy("source", "split", "label_name")
reloaded.show(20)
assert stored.count() == beats.count(), "row count changed on write"
print("round-trip OK")

+------+-----+----------+-----+
|source|split|label_name|count|
+------+-----+----------+-----+
|mitbih| test|         F|  162|
|mitbih| test|         N|18118|
|mitbih| test|         Q| 1608|
|mitbih| test|         S|  556|
|mitbih| test|         V| 1448|
|mitbih|train|         F|  641|
|mitbih|train|         N|72471|
|mitbih|train|         Q| 6431|
|mitbih|train|         S| 2223|
|mitbih|train|         V| 5788|
| ptbdb| full|  abnormal|10506|
| ptbdb| full|    normal| 4046|
+------+-----+----------+-----+



round-trip OK


## What this notebook establishes

| | |
|---|---|
| Beats ingested | **123,998** — 109,446 MIT-BIH + 14,552 PTB |
| Samples per beat | 187 at 125 Hz (1.50 s), zero-padded on the right |
| Storage | 583 MB CSV → **~49 MB Parquet**, ~12× smaller |
| Partitioning | `source` / `split`, so each collection reads independently |

Next: **02 · EDA — MIT-BIH Arrhythmia**.

In [14]:
spark.stop()